# Structured Outputs con la API de OpenAI y Pydantic

En este notebook se implementa un flujo completo para obtener **salidas estructuradas (structured outputs)** de la API de OpenAI, validadas mediante schemas de `pydantic`.

Caso de uso: generación automática de **descripciones de producto optimizadas para SEO** para un e-commerce.

Puntos cubiertos:

1. Preparación del entorno.
2. Definición de los schemas (`pydantic`) que representan el JSON esperado.
3. Llamada a la API de OpenAI solicitando salida estructurada según esos schemas.
4. Validación de la respuesta contra los schemas definidos, con manejo de errores.
5. Caso de uso práctico: generación de descripciones SEO para varios productos ficticios.
6. Análisis de aplicación en un entorno de e-commerce.

## 1. Preparar el entorno de trabajo

In [ ]:
!pip install --upgrade openai pydantic

In [ ]:
import json
from datetime import datetime, timezone
from enum import Enum
from typing import List

from pydantic import BaseModel, Field, ValidationError

from openai import OpenAI

## 2. Definición de los schemas con Pydantic

Definimos tres clases relacionadas:

- `ProductCategory`: enumeración con las categorías de producto soportadas.
- `ProductSEO`: describe el producto (nombre, descripción, palabras clave, puntuación SEO, categoría).
- `GenerationMetadata`: metadatos sobre la generación (fecha, modelo utilizado, idioma).
- `ProductSEOResponse`: combina `ProductSEO` y `GenerationMetadata` en la estructura final que esperamos recibir de la API.

In [ ]:
class ProductCategory(str, Enum):
    ELECTRONICS = "electronics"
    HOME_SERVICES = "home_services"
    FASHION = "fashion"
    HEALTH = "health"
    OTHER = "other"


class ProductSEO(BaseModel):
    name: str = Field(description="Nombre comercial del producto o servicio")
    description: str = Field(description="Descripción del producto optimizada para SEO, entre 50 y 120 palabras")
    keywords: List[str] = Field(description="Lista de 5 a 8 palabras clave relevantes para SEO")
    seo_score: int = Field(ge=0, le=100, description="Puntuación estimada de optimización SEO, de 0 a 100")
    category: ProductCategory = Field(description="Categoría a la que pertenece el producto o servicio")


class GenerationMetadata(BaseModel):
    generated_at: str = Field(description="Fecha y hora de generación en formato ISO 8601")
    model_used: str = Field(description="Nombre del modelo utilizado para generar el contenido")
    language: str = Field(description="Idioma en el que se generó el contenido, por ejemplo 'es' o 'en'")


class ProductSEOResponse(BaseModel):
    product: ProductSEO
    metadata: GenerationMetadata

## 3. Configuración del cliente de OpenAI

Se utiliza una API Key ficticia, tal y como se indica en el ejercicio. Para ejecutar este notebook con resultados reales, sustituye `FAKE_API_KEY` por tu clave real (por ejemplo, mediante una variable de entorno) y desactiva `MOCK_MODE`.

In [ ]:
FAKE_API_KEY = "sk-YOUR_FAKE_API_KEY"

client = OpenAI(api_key=FAKE_API_KEY)

MODEL_NAME = "gpt-4o-2024-08-06"

MOCK_MODE = True

## 4. Llamada a la API con salida estructurada

La función `generate_product_seo` construye un prompt detallado y realiza la petición a la API de OpenAI usando `client.beta.chat.completions.parse`, pasando `ProductSEOResponse` como `response_format`. Esto garantiza que la API devuelva un JSON que respeta exactamente el schema definido.

Como se utiliza una API Key ficticia (`MOCK_MODE = True`), la función simula la respuesta de la API cuando la llamada real falla, para poder demostrar el flujo completo de generación y validación sin necesidad de una clave válida. Con una API Key real y `MOCK_MODE = False`, la función realiza la llamada real a OpenAI.

In [ ]:
def _build_prompt(product_name: str, product_hint: str) -> str:
    return (
        f"Genera una ficha de producto optimizada para SEO para un e-commerce.\n"
        f"Producto: {product_name}.\n"
        f"Contexto adicional: {product_hint}.\n"
        "La descripcion debe tener entre 50 y 120 palabras, en español, "
        "e incluir de 5 a 8 palabras clave relevantes. "
        "Asigna tambien una puntuacion SEO estimada de 0 a 100 y la categoria mas adecuada."
    )


def _mock_response(product_name: str, product_hint: str) -> dict:
    """Genera una respuesta simulada que respeta el schema, usada solo cuando MOCK_MODE=True."""
    return {
        "product": {
            "name": product_name,
            "description": (
                f"Descubre el {product_name.lower()}, disenado para quienes buscan calidad, "
                f"innovacion y comodidad en su dia a dia. {product_hint}. "
                "Ideal para regalar o darte un capricho, combina un diseno elegante con un "
                "rendimiento fiable, pensado para adaptarse a tu estilo de vida y superar tus expectativas."
            ),
            "keywords": [
                product_name.lower(),
                "comprar online",
                "calidad premium",
                "envio rapido",
                "mejor precio",
                "oferta limitada",
            ],
            "seo_score": 82,
            "category": "electronics" if "reloj" in product_name.lower() else "home_services",
        },
        "metadata": {
            "generated_at": datetime.now(timezone.utc).isoformat(),
            "model_used": MODEL_NAME + " (mock)",
            "language": "es",
        },
    }


def generate_product_seo(product_name: str, product_hint: str) -> ProductSEOResponse:
    prompt = _build_prompt(product_name, product_hint)

    if not MOCK_MODE:
        completion = client.beta.chat.completions.parse(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": "Eres un experto en copywriting SEO para e-commerce."},
                {"role": "user", "content": prompt},
            ],
            response_format=ProductSEOResponse,
        )
        return completion.choices[0].message.parsed

    raw_json = _mock_response(product_name, product_hint)
    return ProductSEOResponse.model_validate(raw_json)

## 5. Validación de la respuesta

`ProductSEOResponse.model_validate(...)` valida automáticamente la estructura del JSON recibido contra el schema de `pydantic`. Envolvemos la llamada en un bloque `try/except` para capturar y notificar cualquier error de validación.

In [ ]:
def get_validated_product_seo(product_name: str, product_hint: str):
    try:
        result = generate_product_seo(product_name, product_hint)
        print("Validacion superada. La respuesta cumple con el schema ProductSEOResponse.")
        return result
    except ValidationError as e:
        print("Error de validacion: la respuesta de la API no cumple con el schema esperado.")
        print(e)
        return None
    except Exception as e:
        print(f"Error inesperado al generar o validar la respuesta: {e}")
        return None

## 6. Caso de uso práctico

Probamos el sistema con dos productos ficticios distintos: un **reloj inteligente** y un **servicio de limpieza a domicilio**.

In [ ]:
smartwatch_result = get_validated_product_seo(
    product_name="Reloj Inteligente AuraFit X2",
    product_hint="Monitoriza ritmo cardiaco, sueno y actividad fisica, con bateria de 10 dias",
)

print(json.dumps(smartwatch_result.model_dump(), indent=2, ensure_ascii=False))

In [ ]:
cleaning_service_result = get_validated_product_seo(
    product_name="Servicio de Limpieza Hogar Express",
    product_hint="Limpieza profesional a domicilio, con productos ecologicos y reserva en menos de 5 minutos",
)

print(json.dumps(cleaning_service_result.model_dump(), indent=2, ensure_ascii=False))

Probamos con un tercer producto para verificar que el sistema es consistente y reutilizable con distintos tipos de productos.

In [ ]:
headphones_result = get_validated_product_seo(
    product_name="Auriculares NoiseZero Pro",
    product_hint="Cancelacion activa de ruido, 30 horas de bateria y sonido de alta fidelidad",
)

print(json.dumps(headphones_result.model_dump(), indent=2, ensure_ascii=False))

## 7. Manejo de errores de validación

Para comprobar que el mecanismo de validación funciona correctamente, simulamos una respuesta que **no** cumple con el schema (por ejemplo, `seo_score` fuera de rango y `keywords` con un tipo incorrecto), y verificamos que el sistema lo detecta y lo notifica.

In [ ]:
invalid_raw_json = {
    "product": {
        "name": "Producto de prueba",
        "description": "Descripcion de prueba",
        "keywords": "esto deberia ser una lista, no un string",
        "seo_score": 150,
        "category": "electronics",
    },
    "metadata": {
        "generated_at": datetime.now(timezone.utc).isoformat(),
        "model_used": MODEL_NAME,
        "language": "es",
    },
}

try:
    ProductSEOResponse.model_validate(invalid_raw_json)
    print("La validacion no deberia haber pasado.")
except ValidationError as e:
    print("Error de validacion detectado correctamente:")
    print(e)

## 8. Análisis: aplicación en un entorno de e-commerce

Este flujo de trabajo tiene aplicaciones directas en un entorno de e-commerce real:

- **Generación masiva de fichas de producto**: permite crear automáticamente descripciones optimizadas para SEO para catálogos con miles de productos, reduciendo drásticamente el tiempo de creación de contenido manual.
- **Consistencia estructural garantizada**: al forzar la salida a un schema fijo (`ProductSEOResponse`), se evita que el modelo devuelva texto libre difícil de integrar en una base de datos o un CMS; el JSON resultante puede insertarse directamente en el sistema de gestión de catálogo.
- **Escalabilidad multi-categoría**: gracias al campo `category` (basado en `ProductCategory`), el mismo pipeline puede adaptarse a distintas líneas de negocio (electrónica, moda, servicios del hogar, salud, etc.) sin cambiar la lógica del sistema.
- **Control de calidad automatizado**: el campo `seo_score` permite priorizar qué fichas de producto necesitan revisión humana adicional antes de publicarse, optimizando el trabajo del equipo de marketing.
- **Reducción de errores de integración**: la validación con `pydantic` actúa como una capa de seguridad que detecta respuestas mal formadas antes de que lleguen a producción, evitando fallos en la web o en la app del e-commerce.

En definitiva, combinar *structured outputs* de un LLM con validación mediante `pydantic` permite integrar modelos de lenguaje en pipelines de producción de forma fiable, algo esencial en un entorno de e-commerce donde el contenido debe ser consistente, validado y fácil de automatizar a gran escala.

## Conclusiones

- Se definieron tres schemas relacionados con `pydantic`: `ProductSEO`, `GenerationMetadata` y `ProductSEOResponse`.
- Se implementó una función que realiza una petición a la API de OpenAI solicitando *structured outputs* conformes a `ProductSEOResponse`.
- Se validaron las respuestas contra los schemas definidos, incluyendo un caso de prueba con manejo de errores de validación.
- Se aplicó el sistema a un caso de uso real de e-commerce: generación de descripciones SEO para varios productos ficticios (reloj inteligente, servicio de limpieza, auriculares).
- Se analizó cómo esta implementación puede integrarse en un pipeline de producción de un e-commerce real.